# LLM Evaluation: How Do You Know If Your AI Is Actually Good?

**Author:** Paarth Shah | [LinkedIn](https://www.linkedin.com/in/paarthshah) | [Portfolio](https://paarthcshah.wixsite.com/paarth-portfolio)

---

## The Problem

Everyone is building AI products. Very few people are measuring whether they actually work.

At AriesView, I worked on enterprise RAG pipelines where the stakes were high: if the AI gave a wrong answer, it affected real business decisions. That experience taught me that **evaluation is not an afterthought. It is the product.**

This notebook walks through how to evaluate LLM output quality using three core metrics:

- **Faithfulness**: Did the AI's answer actually come from the source document, or did it make things up?
- **Answer Relevance**: Did the AI answer the question that was actually asked?
- **Context Recall**: Did the retrieval system surface the right information in the first place?

We use the [RAGAS framework](https://docs.ragas.io/) to measure all three.

## Setup

Install the required libraries. You only need to run this once.

In [ ]:
# Install dependencies
# Run this cell first before anything else
!pip install ragas datasets openai -q

In [ ]:
# Import the tools we need
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_recall
from datasets import Dataset
import os

# You need an OpenAI API key for RAGAS to run evaluations
# Replace the string below with your actual key, or set it as an environment variable
os.environ["OPENAI_API_KEY"] = "your-api-key-here"

## The Sample Data

To evaluate an LLM, you need four things for each question:

| Field | What it is |
|-------|------------|
| `question` | The question the user asked |
| `answer` | What the LLM actually said |
| `contexts` | The source chunks the retrieval system pulled |
| `ground_truth` | The correct answer (your benchmark) |

Below is a small example dataset. In a real enterprise deployment, this would be hundreds or thousands of rows.

In [ ]:
# Sample evaluation dataset
# Each entry = one question-answer pair to evaluate

sample_data = {
    "question": [
        "What is the refund policy for enterprise customers?",
        "How long does onboarding typically take?",
        "What integrations does the platform support?"
    ],
    "answer": [
        "Enterprise customers are eligible for a full refund within 30 days of purchase.",
        "Onboarding usually takes 2 to 4 weeks depending on team size.",
        "The platform integrates with Salesforce, HubSpot, and Slack."
    ],
    "contexts": [
        ["Enterprise refund policy: customers may request a full refund within 30 days."],
        ["Standard onboarding is 3 weeks. Enterprise onboarding may vary from 2 to 6 weeks."],
        ["Supported integrations include Salesforce and HubSpot. Slack integration is on the roadmap."]
    ],
    "ground_truth": [
        "Enterprise customers can get a full refund within 30 days.",
        "Onboarding takes 2 to 6 weeks depending on the plan.",
        "The platform currently supports Salesforce and HubSpot. Slack is coming soon."
    ]
}

# Convert to a Dataset object that RAGAS can read
dataset = Dataset.from_dict(sample_data)
print(f"Dataset loaded: {len(dataset)} examples")

## Running the Evaluation

RAGAS scores each metric from 0 to 1. Higher is better.

Notice that example 3 above has a subtle hallucination: the AI said Slack is supported, but the source document says it is only on the roadmap. A good evaluation framework catches exactly this.

In [ ]:
# Run the evaluation across all three metrics
results = evaluate(
    dataset=dataset,
    metrics=[
        faithfulness,       # Did the answer stick to the source?
        answer_relevancy,   # Did it actually answer the question?
        context_recall      # Did retrieval surface the right chunks?
    ]
)

print(results)

## Interpreting the Results

A score below **0.7** on faithfulness is a red flag. It means your AI is regularly going beyond what the source documents say, which in an enterprise context is a trust and compliance problem, not just a quality problem.

In production at AriesView, we used these scores to:

- Set alert thresholds for when outputs needed human review
- Compare retrieval strategies (chunk size, embedding models) against each other
- Report AI quality metrics to stakeholders in business terms, not technical ones

---

## Key Takeaway

Shipping an LLM feature is easy. Knowing whether it is actually working is hard. Evaluation frameworks like RAGAS give you the instrumentation to move from "it seems fine" to "we have data."

If you are building enterprise AI and you are not measuring this, you are flying blind.

---

*Built by Paarth Shah. Questions or feedback? Reach me at paarth.shah@columbia.edu*